# 1단계: 라이브러리 설치
이 단계에서는 KoBERT 모델 학습에 필요한 패키지를 설치합니다.

In [ ]:
!pip install transformers
!pip install 'git+https://github.com/SKTBrain/KoBERT.git#egg=kobert_tokenizer&subdirectory=kobert_hf'

  Cloning https://github.com/SKTBrain/KoBERT.git to /tmp/pip-install-n9bfjvm1/kobert-tokenizer_3279bac4269a4b2d953ac0c316615695
  Running command git clone --filter=blob:none --quiet https://github.com/SKTBrain/KoBERT.git /tmp/pip-install-n9bfjvm1/kobert-tokenizer_3279bac4269a4b2d953ac0c316615695
  Resolved https://github.com/SKTBrain/KoBERT.git to commit fcd729f2f4b37858f333597c0782388ada51eb5f
  Preparing metadata (setup.py) ... done
  Created wheel for kobert_tokenizer: filename=kobert_tokenizer-0.1-py3-none-any.whl size=4633 sha256=0691af31553fe4f0cc84f9e881f886c9e27e3a8285d35b568f3d997b7636900c
  Stored in directory: /tmp/pip-ephem-wheel-cache-h44b65g8/wheels/b7/95/7c/3f38b039ae9422cc4a51702b37e352d0585442cbe60285f1f2
Successfully built kobert_tokenizer


# 2단계: 라이브러리 불러오기
PyTorch, Transformers 등 학습에 필요한 라이브러리를 import 합니다.

In [ ]:
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers.optimization import get_cosine_schedule_with_warmup
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np
import os

# KoBERT 관련 라이브러리
from kobert_tokenizer import KoBERTTokenizer
from transformers import BertModel, BertConfig, AutoTokenizer, AutoModelForSequenceClassification

# 3단계: Google Drive 연결
학습 데이터와 모델 저장을 위해 Google Drive를 마운트합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 4단계: 데이터 경로 설정
CSV 데이터 파일 경로를 지정합니다.

In [ ]:
# 파일 경로 (필요하면 수정)
file1 = "/content/drive/MyDrive/중간평가_이후_모델/all_preprocessed.csv"   # 0/1 섞여 있음 (text,label)
file2 = "/content/drive/MyDrive/중간평가_이후_모델/merged_dialogues.csv"   # 일반 대화 0만, 매우 큼

# 5단계: 데이터 불러오기
보이스피싱 데이터와 일반 대화 데이터를 DataFrame 형태로 읽어옵니다.

In [ ]:
import pandas as pd

# 두 CSV 파일 읽기
mixed = pd.read_csv(file1)
daily = pd.read_csv(file2)

In [ ]:
mixed

,text,label
0,웰컴 더투거래입니다.,1
1,문자 받고 전화드렸는데요.,1
2,"아, 문자 언제 받으셨어요?",1
3,문자는 언제 받으셨어요?,1
4,좀 전에 받으셨어요.,1
...,...,...
1835,고객님. 그러시면 일단 제가 공문서 보내드리고 승인 처리 나시면 바로 전화 드리도록...,1
1836,어. 뭐 상황 처리는 오늘 당일 중에 바로 진행 가능하신 부분 맞으시죠?,1
1837,오늘 해드릴게요.,0
1838,아. 알겠습니다.,1


In [ ]:
daily

,label,text
0,0,집사님 저는 초등학교 학창 시절을 항상 그리워해요.
1,0,왜냐하면 우리가 생각하는 국민학교죠.
2,0,국민학교 때 제가 재밌었는가 봐요.
3,0,시골에서 살았는데
4,0,엄마하고 떨어져서
...,...,...
728276,0,그 사랑은 이루어지지 못했고
728277,0,어 그 마을을 떠나게 됐고. 그래서
728278,0,어 떠난 이후에
728279,0,다시 돌아와서 과거를 회상하는 그런 장면이 나오는데


# 6단계: 데이터 전처리 및 학습 데이터 준비
데이터를 학습 가능한 형태로 정리하고 train/test 데이터를 분리합니다.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
mixed = mixed.rename(columns={mixed.columns[0]: "text", mixed.columns[1]: "label"})
daily = daily.rename(columns={daily.columns[0]: "label", daily.columns[1]: "text"})

In [ ]:
pos_df = mixed[mixed["label"] == 1].copy()
neg_pool = daily[daily["label"] == 0].copy()
n_pos = len(pos_df)
neg_sample = neg_pool.sample(n=n_pos, random_state=42, replace=False)

balanced = pd.concat(
    [pos_df[["text","label"]], neg_sample[["text","label"]]],
    axis=0, ignore_index=True
).sample(frac=1.0, random_state=42).reset_index(drop=True)

train_df, temp_df = train_test_split(
    balanced, test_size=0.2, random_state=42, stratify=balanced["label"]
)
valid_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df["label"]
)

In [ ]:
balanced

,text,label
0,그 자금적인 부분에 대해서 본인이 합법적인 자금이지 또는 불법적인 자금인지 이런 부...,1
1,"아니 페퍼가 접수가 들어간게 아니구요. 페퍼는 심의 있는 부분들 2,300만원 심의...",1
2,알고 보니까 내가 많이 하고 있던 거였더라고.,0
3,금액이 895만원입니다.,1
4,부결 처리가 예 그렇죠.,1
...,...,...
3025,어 나는 운동하면 굉장히 안 좋아하는 편이라서,0
3026,그래서 비공개 수사라 이 말입니다.,1
3027,제가 본인 조사 마무리해야 되기 때문에 어쨌든 본인이 무고한 피해자 라면은 입증이 ...,1
3028,재녹취 진행할 테니까 들어가지 마시라고.,1


# 7단계: 토크나이저 및 모델 준비
KoBERT 토크나이저와 사전학습 모델을 불러옵니다.

In [ ]:
# ## 5단계: 토크나이저 및 모델 준비 ##

# 사용할 KoBERT 모델 이름 정의
MODEL_NAME = "skt/kobert-base-v1"

# 모델에 맞는 토크나이저 불러오기
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/371k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/244 [00:00<?, ?B/s]

In [ ]:
# ## 6단계: 데이터셋 토큰화 ##

# 텍스트를 토큰화하는 함수 정의
def tokenize_function(examples):
    tokenized_output = tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)
    # 단일 문장 분류의 경우 token_type_ids를 모두 0으로 설정
    tokenized_output["token_type_ids"] = [[0] * 128 for _ in range(len(examples["text"]))]
    return tokenized_output

# Pandas DataFrame을 Hugging Face Dataset으로 변환
train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(valid_df)
test_dataset = Dataset.from_pandas(test_df)

print("✅ Hugging Face 데이터셋으로 변환 완료")

# pandas에서 변환 시 자동으로 생성될 수 있는 불필요한 index 컬럼을 제거합니다.
if '__index_level_0__' in train_dataset.column_names:
    train_dataset = train_dataset.remove_columns(['__index_level_0__'])
if '__index_level_0__' in valid_dataset.column_names:
    valid_dataset = valid_dataset.remove_columns(['__index_level_0__'])
if '__index_level_0__' in test_dataset.column_names:
    test_dataset = test_dataset.remove_columns(['__index_level_0__'])

✅ Hugging Face 데이터셋으로 변환 완료


# 8단계: 데이터 토큰화
텍스트 데이터를 BERT 입력 형식으로 변환합니다.

In [ ]:
# 데이터셋에 토크나이저 적용
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_valid = valid_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

# 모델 학습에 필요한 컬럼만 남기고 정리
tokenized_train = tokenized_train.remove_columns(["text"])
tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_train.set_format("torch")

tokenized_valid = tokenized_valid.remove_columns(["text"])
tokenized_valid = tokenized_valid.rename_column("label", "labels")
tokenized_valid.set_format("torch")

tokenized_test = tokenized_test.remove_columns(["text"])
tokenized_test = tokenized_test.rename_column("label", "labels")
tokenized_test.set_format("torch")

print("✅ 데이터 토큰화 및 포맷 설정 완료")

Map:   0%|          | 0/2424 [00:00<?, ? examples/s]

Map:   0%|          | 0/303 [00:00<?, ? examples/s]

Map:   0%|          | 0/303 [00:00<?, ? examples/s]

✅ 데이터 토큰화 및 포맷 설정 완료


# 9단계: DataLoader 생성
배치 단위 학습을 위한 DataLoader를 생성합니다.

In [ ]:
# ## 7단계: 데이터로더 생성 ##
batch_size = 32  # 배치 사이즈

train_dataloader = DataLoader(tokenized_train, shuffle=True, batch_size=batch_size)
eval_dataloader = DataLoader(tokenized_valid, batch_size=batch_size)
test_dataloader = DataLoader(tokenized_test, batch_size=batch_size)

print("✅ 데이터로더 생성 완료")


✅ 데이터로더 생성 완료


In [ ]:
# GPU 사용 가능 여부 확인 및 장치 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


# 10단계: 모델 및 옵티마이저 설정
학습에 사용할 모델, Optimizer, Scheduler를 설정합니다.

In [ ]:
# ## 8단계: 모델, 옵티마이저, 스케줄러 설정 ##
# 사전 학습된 KoBERT 모델 불러오기 (분류 문제용)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

# 옵티마이저 (AdamW) 설정
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

# 학습률 스케줄러 설정
num_epochs = 5  # 학습 에폭 수
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=0, num_training_steps=num_training_steps
)

print("✅ 모델 및 학습 설정 완료. 훈련을 시작합니다...")

config.json:   0%|          | 0.00/535 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/369M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at skt/kobert-base-v1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ 모델 및 학습 설정 완료. 훈련을 시작합니다...


# 11단계: 모델 학습
에폭 단위로 모델을 학습하고 검증을 진행합니다.

In [ ]:
# ## 9단계: 모델 학습 및 검증 ##

progress_bar = tqdm(range(num_training_steps))

for epoch in range(num_epochs):
    # --- 학습 모드 ---
    model.train()
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

    # --- 평가 모드 ---
    model.eval()
    all_preds = []
    all_labels = []
    for batch in eval_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)

        logits = outputs.logits
        predictions = torch.argmax(logits, dim=-1)
        all_preds.extend(predictions.cpu().numpy())
        all_labels.extend(batch["labels"].cpu().numpy())

    # ==================== 🔥 수정된 부분 (검증) 🔥 ====================
    # 검증 성능 계산 시 모든 지표 추가
    eval_accuracy = accuracy_score(all_labels, all_preds)
    eval_precision = precision_score(all_labels, all_preds)
    eval_recall = recall_score(all_labels, all_preds)
    eval_f1 = f1_score(all_labels, all_preds)
    print(f"\n🚀 Epoch {epoch+1} | Validation Accuracy: {eval_accuracy:.4f} | Precision: {eval_precision:.4f} | Recall: {eval_recall:.4f} | F1: {eval_f1:.4f}")

  0%|          | 0/380 [00:00<?, ?it/s]


🚀 Epoch 1 | Validation Accuracy: 0.8020 | Precision: 0.7584 | Recall: 0.8882 | F1: 0.8182

🚀 Epoch 2 | Validation Accuracy: 0.8020 | Precision: 0.7584 | Recall: 0.8882 | F1: 0.8182

🚀 Epoch 3 | Validation Accuracy: 0.8020 | Precision: 0.7584 | Recall: 0.8882 | F1: 0.8182

🚀 Epoch 4 | Validation Accuracy: 0.8020 | Precision: 0.7584 | Recall: 0.8882 | F1: 0.8182

🚀 Epoch 5 | Validation Accuracy: 0.8020 | Precision: 0.7584 | Recall: 0.8882 | F1: 0.8182


# 12단계: 최종 평가
테스트 데이터로 최종 성능을 평가합니다.

In [ ]:
# ## 10단계: 최종 테스트 ##
print("\n✅ 학습 완료! 최종 테스트를 진행합니다...")
model.eval()
all_preds = []
all_labels = []

for batch in test_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)
    all_preds.extend(predictions.cpu().numpy())
    all_labels.extend(batch["labels"].cpu().numpy())

# ==================== 🔥 수정된 부분 (최종 결과) 🔥 ====================
# 최종 성능 계산 시 모든 지표 추가
test_accuracy = accuracy_score(all_labels, all_preds)
test_precision = precision_score(all_labels, all_preds)
test_recall = recall_score(all_labels, all_preds)
test_f1 = f1_score(all_labels, all_preds)

print("\n====== 최종 테스트 결과 ======")
print(f"Accuracy (정확도): {test_accuracy:.4f}")
print(f"Precision (정밀도): {test_precision:.4f}")
print(f"Recall (재현율): {test_recall:.4f}")
print(f"F1 Score: {test_f1:.4f}")
print("==============================")
# ====================================================================


✅ 학습 완료! 최종 테스트를 진행합니다...

====== 최종 테스트 결과 ======
Accuracy (정확도): 0.8185
Precision (정밀도): 0.7927
Recall (재현율): 0.8609
F1 Score: 0.8254


# 13단계: 모델 저장
학습이 완료된 모델과 토크나이저를 저장합니다.

In [ ]:
# ## 11단계: 모델 및 토크나이저 저장 ##

# 저장할 디렉토리(폴더) 이름을 지정합니다.
SAVE_PATH = "/content/drive/MyDrive/kobert_phishing_model"

# 폴더가 없으면 생성
import os
if not os.path.exists(SAVE_PATH):
    os.makedirs(SAVE_PATH)

# 학습된 모델 저장
model.save_pretrained(SAVE_PATH)

# 토크나이저 저장
tokenizer.save_pretrained(SAVE_PATH)

print(f"✅ 모델과 토크나이저가 '{SAVE_PATH}' 경로에 성공적으로 저장되었습니다.")

✅ 모델과 토크나이저가 '/content/drive/MyDrive/kobert_phishing_model' 경로에 성공적으로 저장되었습니다.
